In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from tqdm import tqdm
from pathlib import Path
from data_loader import build_complete_dataset
from window import create_record_windows
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import ParameterGrid

In [2]:
def extract_window_features(window_df, fs, prefix=""):
    features = {}
    columns = [column for column in window_df.columns if column != "Time" ]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
            signal = data[:, i]
            features[f"{prefix}_{col}_mean"] = np.mean(signal)
            features[f"{prefix}_{col}_std"] = np.std(signal)
            features[f"{prefix}_{col}_rms"] = np.sqrt(np.mean(signal**2))
            features[f"{prefix}_{col}_max"] = np.max(signal)
            features[f"{prefix}_{col}_min"] = np.min(signal)
            features[f"{prefix}_{col}_ptp"] = np.ptp(signal)
            features[f"{prefix}_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
            features[f"{prefix}_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
            fft_vals = np.abs(np.fft.rfft(signal))
            freqs = np.fft.rfftfreq(len(signal), d=1 / fs)
            features[f"{prefix}_{col}_fft_energy"] = np.sum(fft_vals**2)
            features[f"{prefix}_{col}_fft_peak_freq"] = freqs[np.argmax(fft_vals)]
    return features

In [3]:
def process_record_features(record, create_record_windows_func):
    acc_windows, gyro_windows, mic_windows = create_record_windows_func(record)
    
    record_features = []
    min_len = min(len(acc_windows), len(gyro_windows), len(mic_windows))
    
    for i in range(min_len):
        features = {}
        features.update(extract_window_features(acc_windows[i], fs=6700, prefix="acc"))
        features.update(extract_window_features(gyro_windows[i], fs=6700, prefix="gyro"))
        features.update(extract_window_features(mic_windows[i], fs=16000, prefix="mic"))
        
        features["segment_id"] = record.metadata.get("segment_id")
        features["split_label"] = record.metadata.get("split_label")
        features["anomaly_label"] = record.metadata.get("anomaly_label")
        features["domain_shift_op"] = record.metadata.get("domain_shift_op")
        features["domain_shift_env"] = record.metadata.get("domain_shift_env")
        record_features.append(features)
        
    return pd.DataFrame(record_features)

In [4]:
def build_feature_dataset(dataset, create_record_windows_func):
    results = []
    for record in tqdm(dataset, desc="Ekstrakcja cech"):
        df_feats = process_record_features(record, create_record_windows_func)
        results.append(df_feats)
    return pd.concat(results, ignore_index=True)

In [5]:
path = Path("../data")
train_path = path / "X_train.csv"
test_path = path / "X_test.csv"

if train_path.exists() and test_path.exists():
    X_train_df = pd.read_csv(train_path)
    X_test_df = pd.read_csv(test_path)
else:
    df = build_complete_dataset()
    X_train_df = build_feature_dataset(df[0], create_record_windows)
    X_test_df = build_feature_dataset(df[1], create_record_windows)
    X_train_df.to_csv(train_path, index=False)
    X_test_df.to_csv(test_path, index=False)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

print(X_train_df.columns.tolist())
print(X_test_df.columns.tolist())


Train: (102984, 75)
Test : (25984, 75)
['acc_A_x [g]_mean', 'acc_A_x [g]_std', 'acc_A_x [g]_rms', 'acc_A_x [g]_max', 'acc_A_x [g]_min', 'acc_A_x [g]_ptp', 'acc_A_x [g]_skew', 'acc_A_x [g]_kurtosis', 'acc_A_x [g]_fft_energy', 'acc_A_x [g]_fft_peak_freq', 'acc_A_y [g]_mean', 'acc_A_y [g]_std', 'acc_A_y [g]_rms', 'acc_A_y [g]_max', 'acc_A_y [g]_min', 'acc_A_y [g]_ptp', 'acc_A_y [g]_skew', 'acc_A_y [g]_kurtosis', 'acc_A_y [g]_fft_energy', 'acc_A_y [g]_fft_peak_freq', 'acc_A_z [g]_mean', 'acc_A_z [g]_std', 'acc_A_z [g]_rms', 'acc_A_z [g]_max', 'acc_A_z [g]_min', 'acc_A_z [g]_ptp', 'acc_A_z [g]_skew', 'acc_A_z [g]_kurtosis', 'acc_A_z [g]_fft_energy', 'acc_A_z [g]_fft_peak_freq', 'gyro_G_x [mdps]_mean', 'gyro_G_x [mdps]_std', 'gyro_G_x [mdps]_rms', 'gyro_G_x [mdps]_max', 'gyro_G_x [mdps]_min', 'gyro_G_x [mdps]_ptp', 'gyro_G_x [mdps]_skew', 'gyro_G_x [mdps]_kurtosis', 'gyro_G_x [mdps]_fft_energy', 'gyro_G_x [mdps]_fft_peak_freq', 'gyro_G_y [mdps]_mean', 'gyro_G_y [mdps]_std', 'gyro_G_y [mdps]_

In [6]:
X_train_source = X_train_df[X_train_df["split_label"] == "Normal_Source_Train"].copy()
X_train_target = X_train_df[X_train_df["split_label"] == "Normal_Target_Train"].copy()

X_val = X_test_df[X_test_df["split_label"].isin(["Normal_Source_Test", "Anomaly_Source_Test"])].copy() # zbior walidacyjny

X_test = X_test_df[X_test_df["split_label"].isin(["Normal_Target_Test", "Anomaly_Target_Test"])].copy() # to nasz zbiór testowy

In [7]:
metadata_cols = ["segment_id", "anomaly_label", "domain_shift_op", "domain_shift_env"]

acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]

In [8]:
X_train_acc = X_train_df[acc_features].fillna(0)
X_test_acc = X_test[acc_features].fillna(0)

X_train_mic = X_train_df[mic_features].fillna(0)
X_test_mic = X_test[mic_features].fillna(0)

X_train_gyro = X_train_df[gyro_features].fillna(0)
X_test_gyro = X_test[gyro_features].fillna(0)

X_train_source_acc = X_train_source[acc_features].fillna(0)
X_train_source_mic = X_train_source[mic_features].fillna(0)
X_train_source_gyro = X_train_source[gyro_features].fillna(0)

X_train_target_acc = X_train_target[acc_features].fillna(0)
X_train_target_mic = X_train_target[mic_features].fillna(0)
X_train_target_gyro = X_train_target[gyro_features].fillna(0)

X_val_acc = X_val[acc_features].fillna(0)
X_val_mic = X_val[mic_features].fillna(0)
X_val_gyro = X_val[gyro_features].fillna(0)

X_test_acc = X_test[acc_features].fillna(0)
X_test_mic = X_test[mic_features].fillna(0)
X_test_gyro = X_test[gyro_features].fillna(0)


In [13]:
def find_best_threshold(y_true, scores):
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)
    thresholds = np.percentile(scores, np.linspace(0, 100, 200))
    best_threshold = thresholds[0]
    best_f1 = -1
    for threshold in thresholds:
        y_pred = (scores >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

In [ ]:
def tune_isolation_forest(
    X_train,
    X_val,
    train_df,
    val_df,
    sensor_type,
    param_grid):

    results = []
    y_val = (val_df.groupby("segment_id")["anomaly_label"].first().eq("loosescrewsA").astype(int))

    for params in ParameterGrid(param_grid):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        model = IsolationForest(
            n_estimators=params["n_estimators"],
            max_samples=params["max_samples"],
            max_features=params["max_features"],
            bootstrap=params["bootstrap"],
            contamination="auto",
            random_state=42,
            n_jobs=-1)

        model.fit(X_train_scaled)

        val_scores = -model.score_samples(X_val_scaled)

        val_results = pd.DataFrame({"segment_id": val_df["segment_id"].values, "score": val_scores})
        segment_scores = (val_results.groupby("segment_id").agg(score=("score", "max")).reset_index())
        segment_scores = segment_scores.merge(y_val.rename("y_true"), left_on="segment_id", right_index=True, how="inner")

        threshold, best_f1 = find_best_threshold(segment_scores["y_true"].values, segment_scores["score"].values)
        y_pred = (segment_scores["score"] >= threshold).astype(int)

        precision = precision_score(segment_scores["y_true"], y_pred, zero_division=0)
        recall = recall_score(segment_scores["y_true"], y_pred, zero_division=0)

        accuracy = accuracy_score(segment_scores["y_true"], y_pred)
        roc_auc = roc_auc_score(segment_scores["y_true"], segment_scores["score"])

        results.append({"sensor": sensor_type,
            **params,
            "threshold": threshold,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall, 
            "ROC AUC": roc_auc,
            "F1": best_f1})
        
    results_df = pd.DataFrame(results)

    best_row = results_df.loc[results_df["F1"].idxmax()]

    best_params = {
        "n_estimators": int(best_row["n_estimators"]),
        "max_samples": best_row["max_samples"],
        "max_features": best_row["max_features"],
        "bootstrap": bool(best_row["bootstrap"]),
        }

    best_threshold = best_row["threshold"]
    print(f"Sensor: {sensor_type}")
    print("Parameters:")
    print(best_params)

    print(f"Threshold : {best_threshold:.6f}")
    print(f"Accuracy  : {best_row['Accuracy']:.4f}")
    print(f"Precision : {best_row['Precision']:.4f}")
    print(f"Recall    : {best_row['Recall']:.4f}")
    print(f"ROC AUC   : {best_row['ROC AUC']:.4f}")
    print(f"F1        : {best_row['F1']:.4f}")


    print("All configurations:")
    print(results_df.sort_values("F1", ascending=False).head(10))

    return (best_params, best_threshold, results_df)

In [20]:
param_grid = {
    "n_estimators": [50, 100, 200, 300, 400],
    "max_samples": [256, 512, 1024, 2048],
    "max_features": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "bootstrap": [False, True]
}

In [21]:
best_params_acc, threshold_acc, tuning_acc = tune_isolation_forest(
    X_train_source_acc,
    X_val_acc,
    X_train_source,
    X_val,
    "ACC source",
    param_grid
)

Sensor: ACC source
Parameters:
{'n_estimators': 50, 'max_samples': 1024, 'max_features': 0.8, 'bootstrap': False}
Threshold : 0.526835
Accuracy  : 0.7716
Precision : 0.7203
Recall    : 0.8879
ROC AUC   : 0.7525
F1        : 0.7954
All configurations:
         sensor  bootstrap  max_features  max_samples  n_estimators  \
70   ACC source      False           0.8         1024            50   
155  ACC source       True           0.6         2048            50   
135  ACC source       True           0.5         2048            50   
10   ACC source      False           0.5         1024            50   
140  ACC source       True           0.6          256            50   
78   ACC source      False           0.8         2048           300   
20   ACC source      False           0.6          256            50   
115  ACC source      False           1.0         2048            50   
21   ACC source      False           0.6          256           100   
35   ACC source      False           0.6

In [22]:
best_params_mic, threshold_mic, tuning_mic = tune_isolation_forest(
    X_train_source_mic,
    X_val_mic,
    X_train_source,
    X_val,
    "MIC source",
    param_grid
)

Sensor: MIC source
Parameters:
{'n_estimators': 50, 'max_samples': 256, 'max_features': 0.6, 'bootstrap': False}
Threshold : 0.436688
Accuracy  : 0.5690
Precision : 0.5396
Recall    : 0.9397
ROC AUC   : 0.5951
F1        : 0.6855
All configurations:
         sensor  bootstrap  max_features  max_samples  n_estimators  \
20   MIC source      False           0.6          256            50   
140  MIC source       True           0.6          256            50   
221  MIC source       True           1.0          256           100   
101  MIC source      False           1.0          256           100   
141  MIC source       True           0.6          256           100   
220  MIC source       True           1.0          256            50   
100  MIC source      False           1.0          256            50   
21   MIC source      False           0.6          256           100   
178  MIC source       True           0.7         2048           300   
73   MIC source      False           0.8 

In [23]:
best_params_gyro, threshold_gyro, tuning_gyro = tune_isolation_forest(
    X_train_source_gyro,
    X_val_gyro,
    X_train_source,
    X_val,
    "GYRO source",
    param_grid
)

Sensor: GYRO source
Parameters:
{'n_estimators': 200, 'max_samples': 1024, 'max_features': 0.7, 'bootstrap': False}
Threshold : 0.447177
Accuracy  : 0.5259
Precision : 0.5136
Recall    : 0.9741
ROC AUC   : 0.5889
F1        : 0.6726
All configurations:
          sensor  bootstrap  max_features  max_samples  n_estimators  \
52   GYRO source      False           0.7         1024           200   
53   GYRO source      False           0.7         1024           300   
117  GYRO source      False           1.0         2048           200   
51   GYRO source      False           0.7         1024           100   
173  GYRO source       True           0.7         1024           300   
193  GYRO source       True           0.8         1024           300   
17   GYRO source      False           0.5         2048           200   
194  GYRO source       True           0.8         1024           400   
72   GYRO source      False           0.8         1024           200   
58   GYRO source      False 

In [ ]:
def run_isolation_forest(X_train, X_test, test_df, sensor_type, best_params, threshold):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = IsolationForest(
        n_estimators=best_params["n_estimators"],
        max_samples=best_params["max_samples"],
        max_features=best_params["max_features"],
        bootstrap=best_params["bootstrap"],
        contamination="auto",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_scaled)

    test_scores = -model.score_samples(X_test_scaled)
    
    results = pd.DataFrame({"segment_id": test_df["segment_id"].values,
        "anomaly_label": test_df["anomaly_label"].values,
        "score": test_scores})

    segment_scores = (results.groupby("segment_id").agg(score=("score", "max"), anomaly_label=("anomaly_label", "first")).reset_index())
    segment_scores["y_true"] = (segment_scores["anomaly_label"] == "loosescrewsA").astype(int)
    segment_scores["y_pred"] = (segment_scores["score"] >= threshold).astype(int)

    y_true = segment_scores["y_true"]
    y_pred = segment_scores["y_pred"]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    roc_auc = roc_auc_score(y_true, segment_scores["score"]) 

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"{sensor_type}")
    print(f"Threshold: {threshold:.4f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")
    print(cm)

    metrics = {"Sensors": sensor_type,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]}

    return metrics, segment_scores, model